In [1]:
import numpy as np
import math

We solved the non-diagonal rectangles version in [problem 85][1], which we shows for an $m \times n$ grid, the number of rectangles $N$ is
$$
N = {m+1 \choose 2} {n+1 \choose 2}
$$

Adding the diagonals adds a lot of challenges. The problem is that diagonals are not uniform in size--the maximum size increases then shrinks as we move left to right along the diagonal. In this case we approach using dynamic programming.

Define $R(n) = {n+1 \choose 2}$ be the number of non-diagonal rectangles in an $1 \times n$ grid (a "row"). Let $d(m,n)$ be the number of rectangles in an $m \times n$ grid with cross-hatches. It is obvious that $d(m, n) = d(n, m)$ (since they are transposes of each other) so we will solve cases where $m \geq n$ (rows $\geq$ columns). Also, it is obvious that $d(1,1) = 1$ can be initialized, since no diagonal squares are possible. Additionally, from a few examples, you can clearly see that $d(1, n) = R(n) + n - 1$.

Imagine you have solved $d(m-1,n)$ for a $(m-1) \times n$ grid (with $m-1 \geq n$), and you add one row to it to make it $m \times n$ (so $m > n$). The number of non-diagonal rectangles added can be calculated as $R(n) \cdot m$ since for each height h \in \{1,2,3,\dots,m\}$ you can fit $R(n)$ rectangles in the grid of size $h \times n$. 

To get the number of diagonal rectangles, think about the new vertices for rectangles that you added (since you have found $d(m-1, n)$ already, you only need to worry about the new vertices you added). Ignoring the very bottom left corner and very bottom right corner (since no rectangles can be formed there), these have to fall on the crossing points in the center of a square (on the last row that you added) or on the bottom border. Index these from left to right starting from $1$, and since there are a total of $2n + 1$ points, but we subtract out the $2$ extreme corners, we are left with an index from $1$ to $2n - 1$. For each of these crossing points, let it be the "bottom-right" vertex of a rectangle. After trying a few, you will notice if you go up and left from any vertex $i$, it will always be length $i$ and if you go up and right from vertex $i$, it will always be $2n - i$. And because each vertex contains at most $1$ whole square, the area of the rectangle will give you the number of new rectangles (since any new rectangle will have to include that square/half-square). Additionally, because of symmetry, we can stop at the halfway point, namely when we are at index $n$, where the new area to add is a square with sides $n$. We just have to multiply by two for all the ones before the halfway point, but don't multiply two for the $n$ index since there's only one square in the middle!

Now the last case is when $m = n$, so you do the same exercise, but now notice that for each of the odd indices, the diagonal rectangles are $1$ half-square short! So simply do $-1$ on the odd indices when you are looking at a square grid $n \times n$.

From here, you can generate all possible grids by noting that $d(m,n) = d(n,m)$.

This gives a final formula that 
$$
d(m, n) = d(m-1, n) + R(n) \cdot m + n^2 + \sum_{i=1}^{n-1} 2 \cdot (2n - i) - \begin{cases}
2 \cdot \lceil \frac{n}{2} \rceil & m=n \\
0 & m > n
\end{cases}
$$

[1]: https://projecteuler.net/problem=85

In [6]:
fact = [1]
for i in range(1,1000):
    fact.append(fact[-1]*i)

24

In [9]:
def choose(n,k):
    return fact[n] // (fact[n-k] * fact[k])

def r(m,n):
    return choose(m+1, 2)*choose(n+1, 2)

In [142]:
d = [[0 for _ in range(51)] for _ in range(51)]
d[1][1] = 1

# row 0 is just filled with 0s and so dynamic program starts at 1,1
for i in range(1,len(d)):
    d[i][1] = r(i,1) + i - 1
    d[1][i] = r(1,i) + i - 1

In [143]:
for n in range(2, len(d)):
    rects = d[n-1][n]
    for m in range(n, len(d)):
        # vertical and horizontals are R(n) * m
        straights = r(1,n)*m

        # diags need to be calculated using lattice points
        diags = 0

        square_penalty = 1 if m == n else 0

        # area = # of new diag rectangles, since the lattice point is required to be a part of it
        for i in range(1,n):
            # only go half way, but double as you go, due to the symmetry
            diags += 2*(i*(2*n - i) - square_penalty*(i % 2))
        
        # for final area, it's a square, and it doesnt need to be doubled
        diags += n*n - square_penalty*(n % 2)

        rects += straights + diags
        
        d[m][n] = rects
        d[n][m] = rects

print(np.sum(np.array(d)[:48, :44]))

846910284
